In [1]:
from pathlib import Path
import pandas as pd


In [2]:
TABLES = Path("../04_outputs/tables")

cross_1 = pd.read_csv(TABLES / "cross_ben_sarc_binary_to_banglasarc3_binary_results.csv")
cross_2 = pd.read_csv(TABLES / "cross_banglasarc3_binary_to_ben_sarc_binary_results.csv")
bert_df = pd.read_csv(TABLES / "banglabert_binary_summary.csv")

In [3]:
bert_test = bert_df[bert_df["split"] == "test"].copy()

in_domain = bert_test[["dataset", "accuracy", "macro_f1", "f1_binary"]].rename(columns={
    "dataset": "target_dataset",
    "accuracy": "in_domain_accuracy",
    "macro_f1": "in_domain_macro_f1",
    "f1_binary": "in_domain_f1_binary",
})

in_domain

,target_dataset,in_domain_accuracy,in_domain_macro_f1,in_domain_f1_binary
0,banglasarc3_binary,0.735661,0.735290,0.745192
2,banglasarc_binary,0.976562,0.975052,0.968912
4,ben_sarc_binary,0.796412,0.795731,0.783940


In [4]:
cross_df = pd.concat([cross_1, cross_2], ignore_index=True)
cross_df

,model,source_dataset,target_dataset,accuracy,precision_binary,recall_binary,f1_binary,macro_f1,epochs,batch_size,learning_rate,max_length,seed
0,banglabert_cross_dataset,ben_sarc_binary,banglasarc3_binary,0.669576,0.695402,0.603491,0.646195,0.668127,2,8,0.00002,128,42
1,banglabert_cross_dataset,banglasarc3_binary,ben_sarc_binary,0.685257,0.674504,0.716069,0.694665,0.684958,2,8,0.00002,128,42


In [5]:
comparison_df = cross_df.merge(in_domain, on="target_dataset", how="left")

comparison_df["macro_f1_generalization_gap"] = (
    comparison_df["in_domain_macro_f1"] - comparison_df["macro_f1"]
)

comparison_df["accuracy_generalization_gap"] = (
    comparison_df["in_domain_accuracy"] - comparison_df["accuracy"]
)

comparison_df = comparison_df[[
    "source_dataset",
    "target_dataset",
    "accuracy",
    "macro_f1",
    "f1_binary",
    "in_domain_accuracy",
    "in_domain_macro_f1",
    "macro_f1_generalization_gap",
    "accuracy_generalization_gap",
]]

comparison_df

,source_dataset,target_dataset,accuracy,macro_f1,f1_binary,in_domain_accuracy,in_domain_macro_f1,macro_f1_generalization_gap,accuracy_generalization_gap
0,ben_sarc_binary,banglasarc3_binary,0.669576,0.668127,0.646195,0.735661,0.735290,0.067164,0.066085
1,banglasarc3_binary,ben_sarc_binary,0.685257,0.684958,0.694665,0.796412,0.795731,0.110773,0.111154


In [6]:
comparison_df.to_csv(TABLES / "cross_dataset_comparison.csv", index=False)
print(TABLES / "cross_dataset_comparison.csv")

../04_outputs/tables/cross_dataset_comparison.csv
